## Step 1 — Load environment and imports

Load `.env` (for API keys) and import the dataset generator and Gradio.

In [1]:
import os
from dotenv import load_dotenv
import gradio as gr

from dataset_generator import DatasetGenerator, get_models_for_provider, PROVIDERS

load_dotenv(override=True)

True

## Step 2 — Define provider list

Providers use the same OpenAI client with different `base_url` and API key (OpenRouter, OpenAI, Ollama, Gemini).

In [2]:
# Provider list comes from dataset_generator (includes OpenAI, OpenRouter, Ollama, Gemini)
# PROVIDERS is already imported above

## Step 3 — Generate dataset handler

Creates a `DatasetGenerator` for the selected provider, calls the LLM, and exports to CSV or JSON.

In [3]:
def generate_dataset(user_input, provider, model_name, export_format):
    """Build generator for selected provider (OpenAI client under the hood), then generate and export."""
    try:
        generator = DatasetGenerator(provider=provider)
        df, latency = generator.generate(model_name, user_input)
        file_path = generator.export(df, export_format)
        status = f"Success | Latency: {latency:.2f}s"
        return df, file_path, status
    except Exception as e:
        return None, None, f"Error: {str(e)}"

## Step 4 — Update model dropdown when provider changes

When the user switches provider, the Model dropdown is updated with that provider’s model list.

In [4]:
def update_models_for_provider(provider):
    """When provider changes, update model dropdown choices (Week 1 multi-provider pattern)."""
    models = get_models_for_provider(provider)
    return gr.update(choices=models, value=models[0] if models else None)

## Step 5 — Build and launch the Gradio UI

Define inputs (describe dataset, provider, model, export format), outputs (table, file download, status), and wire the Generate button and provider-change logic.

In [5]:
with gr.Blocks(title="Dataset Generator") as demo:
    gr.Markdown("## AI Dataset Generator\n*Uses the OpenAI Python client (Week 1) with OpenAI, OpenRouter, Ollama, or Gemini.*")

    user_input = gr.Textbox(
        label="Describe Dataset",
        placeholder="e.g. 20 rows of airline bookings, or synthetic insurance claims, or e-commerce orders…",
        lines=2,
    )
    provider_dropdown = gr.Dropdown(
        choices=PROVIDERS,
        value="openrouter",
        label="Provider",
    )
    model_dropdown = gr.Dropdown(
        choices=get_models_for_provider("openrouter"),
        value=get_models_for_provider("openrouter")[0],
        label="Model",
    )
    export_dropdown = gr.Dropdown(
        choices=["CSV", "JSON"],
        label="Export Format",
        value="CSV",
    )
    generate_btn = gr.Button("Generate")

    output_table = gr.Dataframe(label="Generated Data")
    file_output = gr.File(label="Download File")
    status_output = gr.Textbox(label="Status")

    provider_dropdown.change(
        update_models_for_provider,
        inputs=[provider_dropdown],
        outputs=[model_dropdown],
    )
    generate_btn.click(
        generate_dataset,
        inputs=[user_input, provider_dropdown, model_dropdown, export_dropdown],
        outputs=[output_table, file_output, status_output],
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
